### Data Reading CSV

In [ ]:
# -----------------------------------------------------------------
# setup to do in any notebook
# -----------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (StructType, StructField, IntegerType, DoubleType, StringType)


import pandas as pd
import matplotlib.pyplot as plt

import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
        .appName("AnshLambdaYouTube")
        .master("local[*]")  # use all cores on machine as workers
        # .config("spark.jars.packages", "com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11") # only if using sqlserver
        .getOrCreate()
)

# -----------------------------------------------------------------
# tutorial starts here
# -----------------------------------------------------------------
df = spark.read.format('csv').option('inferSchema', True).option('header', True).load('BigMartSales.csv')

df.show() # simple txt output, fine most of time
# df.display() # databricks only, gives nice html formatted filter capable table
# df.limit(20).toPandas() # local method using pandas - make sure to filter/reduce num results first, don't load entire DataFrame

### Data Reading JSON

In [ ]:
df_json = spark.read.format('json').option('inferSchema',True)\
    .option('header',True)\
    .option('multiLine',False)\
    .load('drivers.json')

df_json.show()

### Schema Definition

In [ ]:
df.printSchema()

### DDL Schema

In [ ]:
my_ddl_schema = '''
  Item_Identifier STRING,
  Item_Weight STRING,
  Item_Fat_Content STRING,
  Item_Visibility DOUBLE,
  Item_Type STRING,
  Item_MRP DOUBLE,
  Outlet_Identifier STRING,
  Outlet_Establishment_Year INT,
  Outlet_Size STRING,
  Outlet_Location_Type STRING,
  Outlet_Type STRING,
  Item_Outlet_Sales DOUBLE
'''

df = spark.read.format('csv')\
    .schema(my_ddl_schema)\
    .option('header',True)\
    .load('BigMartSales.csv')

df.printSchema()
df.show()

### StructType() Schema

In [ ]:
my_struct_schema = StructType([
    StructField('Item_Identifier', StringType(), True),
    StructField('Item_Weight', StringType(), True),
    StructField('Item_Fat_Content', StringType(), True),
    StructField('Item_Visibility', StringType(), True),
    StructField('Item_Type', StringType(), True),
    StructField('Item_MRP', StringType(), True),
    StructField('Outlet_Identifier', StringType(), True),
    StructField('Outlet_Establishment_Year', StringType(), True),
    StructField('Outlet_Size', StringType(), True),
    StructField('Outlet_Location_Type', StringType(), True),
    StructField('Outlet_Type', StringType(), True),
    StructField('Item_Outlet_Sales', StringType(), True),
])

df = spark.read.format('csv')\
    .schema(my_struct_schema)\
    .option('header',True)\
    .load('BigMartSales.csv')

df.printSchema()
df.show()

### Transformations: SELECT, ALIAS, FILTER/WHERE

In [ ]:
# reload to get the correct schema and data types for comparisons
df = spark.read.format('csv').option('inferSchema', True).option('header', True).load('BigMartSales.csv')

df.select(F.col('Item_Identifier'),F.col('Item_Weight'),F.col('Item_Fat_Content')).show()
df.select(F.col('Item_Identifier').alias('item_id'),F.col('Item_Weight'),F.col('Item_Fat_Content')).show()
df.filter(F.col('Item_Fat_Content') == 'Regular').show()
df.filter((F.col('Item_Type') == 'Soft Drinks') & (F.col('Item_Weight') < 10) ).show()
df.filter((F.col('Outlet_Size').isNull()) & (F.col('Outlet_Location_Type').isin('Tier 1','Tier 2')) ).show()
df.withColumnRenamed('Item_Weight','Item_Wt').show()